In [14]:
import requests
import time
import json
import os

# Constants
API_KEY = "3ef1fc14-2fef-4400-ae57-bfb43b8103d0"  # Replace with your actual API key
BASE_URL = "https://api.companieshouse.gov.uk"
COMPANY_NUMBERS = ["01087941", "04168334", "02019274"]  # Replace with the actual company number

# Rate limiting
REQUEST_COUNT = 0
START_TIME = time.time()

# Function to handle rate limit
def check_rate_limit(request_count, start_time):
    if request_count >= 600:
        elapsed_time = time.time() - start_time
        if elapsed_time < 300:
            time.sleep(300 - elapsed_time)
        request_count = 0
        start_time = time.time()
    return request_count, start_time

# Function to retrieve company profile
def get_company_profile(company_number):
    url = f"{BASE_URL}/company/{company_number}"
    response = requests.get(url, auth=(API_KEY, ''))
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Failed to retrieve company profile: {response.status_code}")

# Function to download a PDF document
def download_pdf(document_url, directory, filename):
    file_path = os.path.join(directory, filename)
    if not os.path.exists(file_path):
        pdf_response = requests.get(document_url, auth=(API_KEY, ''), allow_redirects=True)
        if pdf_response.status_code == 200:
            with open(file_path, 'wb') as pdf_file:
                pdf_file.write(pdf_response.content)
            print(f'PDF downloaded: {filename}')
        else:
            raise Exception(f"Failed to download PDF: {pdf_response.status_code}")
    else:
        print(f'File already exists: {filename}')

# Function to process a company
def process_company(company_number):
    global REQUEST_COUNT, START_TIME
    directory = f"./{company_number}"
    if not os.path.exists(directory):
        os.makedirs(directory)

    company_profile = get_company_profile(company_number)
    print(f"Processing company: {company_profile['company_name']} ({company_number})")
    filing_history_url = BASE_URL + company_profile['links']['filing_history']
    items_per_page = 100
    start_index = 0

    while True:
        REQUEST_COUNT, START_TIME = check_rate_limit(REQUEST_COUNT, START_TIME)
        paginated_url = f"{filing_history_url}?start_index={start_index}&items_per_page={items_per_page}"
        response = requests.get(paginated_url, auth=(API_KEY, ''))
        REQUEST_COUNT += 1
        
        if response.status_code == 200:
            filing_history = response.json()
            for item in filing_history['items']:
                if 'accounts' in item['description'].lower():
                    if 'document_metadata' in item['links']:
                        document_id = item['links']['document_metadata']
                        document_url = document_id
                        REQUEST_COUNT, START_TIME = check_rate_limit(REQUEST_COUNT, START_TIME)
                        doc_response = requests.get(document_url, auth=(API_KEY, ''))
                        REQUEST_COUNT += 1

                        if doc_response.status_code == 200:
                            document_metadata = doc_response.json()
                            download_url = document_metadata['links']['document']
                            filename = f"{item['date']}_{item['description'].replace(' ', '_')}.pdf"
                            download_pdf(download_url, directory, filename)
            start_index += items_per_page
            if start_index >= filing_history['total_count']:
                break
        else:
            raise Exception(f"Failed to retrieve filing history: {response.status_code}")
        time.sleep(1)  # To respect the rate limit for safe measure

# Main logic to process each company
for company_number in COMPANY_NUMBERS:
    try:
        process_company(company_number)
    except Exception as e:
        print(f"Error processing company {company_number}: {e}")

Processing company: A.& J.SCOTT LIMITED (01087941)
PDF downloaded: 2023-11-24_accounts-with-accounts-type-full.pdf
PDF downloaded: 2022-05-06_accounts-with-accounts-type-full.pdf
PDF downloaded: 2021-04-06_accounts-with-accounts-type-full.pdf
PDF downloaded: 2020-09-25_accounts-with-accounts-type-full.pdf
PDF downloaded: 2019-05-13_accounts-with-accounts-type-full.pdf
PDF downloaded: 2018-08-01_accounts-with-accounts-type-full.pdf
PDF downloaded: 2017-05-24_accounts-with-accounts-type-small.pdf
PDF downloaded: 2016-08-26_accounts-with-accounts-type-full.pdf
PDF downloaded: 2015-06-11_accounts-with-accounts-type-medium.pdf
PDF downloaded: 2014-05-13_accounts-with-accounts-type-medium.pdf
PDF downloaded: 2013-03-14_accounts-with-accounts-type-medium.pdf
PDF downloaded: 2012-03-06_accounts-with-accounts-type-medium.pdf
PDF downloaded: 2011-03-03_accounts-with-accounts-type-medium.pdf
PDF downloaded: 2010-10-01_accounts-with-accounts-type-medium.pdf
PDF downloaded: 2008-10-08_accounts-with

# BigTech

In [28]:
API_KEY = "3ef1fc14-2fef-4400-ae57-bfb43b8103d0"
BASE_URL = "https://api.companieshouse.gov.uk"
COMPANY_NUMBERS = ["01087941", "04168334", "02019274"]

In [29]:
import requests

class APIClient:
    def __init__(self):
        self.session = requests.Session()
        self.session.auth = (API_KEY, '')
    
    def get(self, url):
        return self.session.get(url)
    
    def get_company_profile(self, company_number):
        url = f"{BASE_URL}/company/{company_number}"
        return self.get(url)
    
    def get_filing_history_page(self, company_number, start_index=0, items_per_page=100):
        url = f"{BASE_URL}/company/{company_number}/filing-history?start_index={start_index}&items_per_page={items_per_page}"
        return self.get(url)

    
    def get_document_metadata(self, url):
        return self.get(url)


In [30]:
import time

class RateLimiter:
    def __init__(self, limit, period):
        self.requests = 0
        self.limit = limit
        self.period = period
        self.start_time = time.time()
    
    def check(self):
        if self.requests >= self.limit:
            elapsed_time = time.time() - self.start_time
            if elapsed_time < self.period:
                time.sleep(self.period - elapsed_time)
            self.requests = 0
            self.start_time = time.time()
        self.requests += 1


In [31]:

import os
import requests

class FileManager:
    def __init__(self, directory):
        self.directory = directory
        os.makedirs(directory, exist_ok=True)
    
    def download_pdf(self, document_url, filename):
        file_path = os.path.join(self.directory, filename)
        if not os.path.exists(file_path):
            # Adding authentication to the request
            response = requests.get(document_url, auth=(API_KEY, ''), allow_redirects=True)
            if response.status_code == 200:
                with open(file_path, 'wb') as pdf_file:
                    pdf_file.write(response.content)
                print(f'PDF downloaded: {filename}')
            else:
                raise Exception(f"Failed to download PDF: {response.status_code}")
        else:
            print(f'File already exists: {filename}')



In [32]:

def process_company(company_number):
    client = APIClient()
    limiter = RateLimiter(600, 300)
    manager = FileManager(f"./{company_number}")
    
    response = client.get_company_profile(company_number)
    if response.status_code == 200:
        company_profile = response.json()
        print(f"Processing company: {company_profile['company_name']} ({company_number})")
        
        start_index = 0
        more_pages = True
        while more_pages:
            limiter.check()
            history_response = client.get_filing_history_page(company_number, start_index)
            if history_response.status_code == 200:
                filing_history = history_response.json()
                for item in filing_history['items']:
                    if 'accounts' in item['description'].lower():
                        print(f"Accounts related document found: {item['description']}")
                        doc_metadata_response = client.get_document_metadata(item['links']['document_metadata'])
                        if doc_metadata_response.status_code == 200:
                            doc_metadata = doc_metadata_response.json()
                            download_url = doc_metadata['links']['document']
                            filename = f"{item['date']}_{item['description'].replace(' ', '_')}.pdf"
                            manager.download_pdf(download_url, filename)
                        else:
                            print("Failed to retrieve document metadata")
                
                start_index += len(filing_history['items'])
                more_pages = start_index < filing_history['total_count']
            else:
                raise Exception("Failed to retrieve filing history")
    else:
        raise Exception("Failed to retrieve company profile")

if __name__ == "__main__":
    for number in COMPANY_NUMBERS:
        try:
            process_company(number)
        except Exception as e:
            print(f"Error processing company {number}: {e}")


Processing company: A.& J.SCOTT LIMITED (01087941)
Accounts related document found: accounts-with-accounts-type-full
PDF downloaded: 2023-11-24_accounts-with-accounts-type-full.pdf
Accounts related document found: accounts-with-accounts-type-full
PDF downloaded: 2022-05-06_accounts-with-accounts-type-full.pdf
Accounts related document found: accounts-with-accounts-type-full
PDF downloaded: 2021-04-06_accounts-with-accounts-type-full.pdf
Accounts related document found: accounts-with-accounts-type-full
PDF downloaded: 2020-09-25_accounts-with-accounts-type-full.pdf
Accounts related document found: accounts-with-accounts-type-full
PDF downloaded: 2019-05-13_accounts-with-accounts-type-full.pdf
Accounts related document found: accounts-with-accounts-type-full
PDF downloaded: 2018-08-01_accounts-with-accounts-type-full.pdf
Accounts related document found: accounts-with-accounts-type-small
PDF downloaded: 2017-05-24_accounts-with-accounts-type-small.pdf
Accounts related document found: acco